# Regression - Monetary Total Prediction

Predict customer monetary value and compare regression models.

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROCESSED_DATA_PATH = Path('../data/processed/retail_customers_cleaned.csv')
REPORTS_DIR = Path('../reports')
MODEL_DIR = Path('../models')

In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
target = 'MonetaryTotal'

X = df.drop(columns=[target])
y = df[target]
X = X.drop(columns=['Churn'], errors='ignore')
X = X.drop(columns=['MonetaryAvg', 'MonetaryStd', 'MonetaryMin', 'MonetaryMax', 'MonetaryPerDay', 'AvgBasketValue'], errors='ignore')

X.shape, y.shape

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encoder', OneHotEncoder(handle_unknown='ignore'))]), categorical_features),
])

In [ ]:
models = [
    ('Linear Regression', LinearRegression()),
    ('Random Forest Regressor', RandomForestRegressor(n_estimators=200, random_state=42)),
]

results = []
for name, model in models:
    pipeline = Pipeline([('preprocessor', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_test, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_test, y_pred)),
        'R2': r2_score(y_test, y_pred),
    })

pd.DataFrame(results).sort_values('RMSE')

In [ ]:
display(pd.read_csv(REPORTS_DIR / 'regression_results.csv').sort_values('RMSE'))
joblib.load(MODEL_DIR / 'best_regression_model.joblib')